# 01 - Synthetic Data Generation

## 项目背景

本项目模拟一个对话式AI产品（类似ChatGPT/Claude）的用户行为数据分析流水线。由于没有真实产品数据，第一步是构造一套结构合理、贴近现实的合成数据集，为后续分析提供基础。

## 本notebook的目标

生成三张核心数据表，模拟30天内500名用户与AI产品的交互记录：

| 表名 | 内容 | 关键字段 |
|------|------|---------|
| `users.csv` | 用户画像 | user_id, user_type, signup_date |
| `conversations.csv` | 对话记录 | conversation_id, user_id, session_id, timestamp, intent, query, response |
| `events.csv` | 行为事件流 | event_id, conversation_id, event_type, event_timestamp, followup_text |

## 流程与设计思路

**Step 1: 用户生成**
- 500名用户分为三类：heavy（30%）、casual（50%）、churned（20%）
- 不同类型的用户在后续步骤中会表现出不同的活跃频率和留存行为

**Step 2: 意图场景与行为概率配置**
- 定义6个对话意图场景（代码生成、知识问答、创意写作、翻译、闲聊、数据分析）
- 每个场景预设不同的用户行为概率分布，例如代码场景retry率高（30%），知识问答dislike率高（25%）
- 这些差异是刻意设计的，目的是让后续分析能发现有意义的cross-scenario patterns

**Step 3: 对话记录生成**
- 根据用户类型决定活跃天数和日均对话量（heavy用户日均3-8条，churned用户仅前7天活跃）
- 时间戳按真实用户作息分布（上午10-11点、晚间20-21点为高峰）
- 同一天的对话归入同一个session_id

**Step 4: 文本填充**
- 为每条对话填充query和response文本模板
- response中按一定概率混入"问题回复"（拒答、幻觉、答非所问等），且该概率与场景的dislike+retry率正相关
- 注意：不预打任何归因标签，保持与真实埋点数据一致——归因分析留给后续notebook

**Step 5: 事件流生成**
- 采用事件流模型而非单标签模型：一条对话可能触发多个事件（如先retry再点赞）
- 沉默用户不产生任何事件记录（与真实埋点一致）
- 追问文本区分"探索式"和"纠错式"两种风格，但不打标签，留给后续分析判断

## 输出

三张csv文件保存在 `data/` 目录下，供后续notebook使用。

# 01 - Synthetic Data Generation

## Background

This project builds a user behavior analytics pipeline for a conversational AI product (similar to ChatGPT/Claude). Since no real production data is available, the first step is to generate a synthetic dataset that is structurally realistic and analytically meaningful.

## Objective

Generate three core tables simulating 30 days of interaction data from 500 users:

| Table | Content | Key Fields |
|-------|---------|------------|
| `users.csv` | User profiles | user_id, user_type, signup_date |
| `conversations.csv` | Conversation logs | conversation_id, user_id, session_id, timestamp, intent, query, response |
| `events.csv` | Behavioral event stream | event_id, conversation_id, event_type, event_timestamp, followup_text |

## Pipeline & Design Rationale

**Step 1: User Generation**
- 500 users assigned to three segments: heavy (30%), casual (50%), churned (20%)
- Each segment exhibits distinct activity frequency and retention patterns in subsequent steps

**Step 2: Intent & Behavior Probability Configuration**
- 6 conversation intent categories defined: code generation, knowledge QA, creative writing, translation, chitchat, data analysis
- Each intent has a tailored behavior probability distribution (e.g., code generation has a 30% retry rate; knowledge QA has a 25% dislike rate)
- These differences are intentional — designed so that downstream analysis can surface meaningful cross-scenario patterns

**Step 3: Conversation Generation**
- Active days and daily conversation volume are driven by user type (heavy users: 3–8/day; churned users: active only in the first 7 days)
- Timestamps follow a realistic hourly distribution (peaks at 10–11 AM and 8–9 PM)
- Conversations on the same day are grouped under a shared session_id

**Step 4: Text Population**
- Each conversation is populated with query and response templates per intent
- A proportion of responses are intentionally "bad" (refusals, hallucinations, off-topic answers), with the probability tied to each intent's dislike + retry rate
- Crucially, no attribution labels are attached — this mirrors real logging pipelines where root-cause labeling is a separate analytical step

**Step 5: Event Stream Generation**
- Uses an event stream model, not a single-label model: one conversation can trigger multiple events (e.g., retry → like)
- Silent users produce no event records (consistent with real instrumentation)
- Follow-up texts are written in two distinct styles — exploratory ("Can you elaborate?") and corrective ("That's not what I asked") — but left unlabeled for downstream classification

## Output

Three CSV files saved to `data/`, consumed by subsequent notebooks in the pipeline.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# 固定随机种子，保证可复现
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# 基础参数
N_USERS = 500
DATE_START = datetime(2025, 3, 1)
DATE_END = datetime(2025, 3, 31)
OUTPUT_DIR = '../data'

print(f'时间范围: {DATE_START.date()} ~ {DATE_END.date()}')
print(f'模拟用户数: {N_USERS}')

时间范围: 2025-03-01 ~ 2025-03-31
模拟用户数: 500


In [2]:
# === 用户表生成 ===

# 用户类型分布：重度30%、轻度50%、流失20%
user_types = np.random.choice(
    ['heavy', 'casual', 'churned'],
    size=N_USERS,
    p=[0.3, 0.5, 0.2]
)

# 注册时间：大部分在产品上线前就注册了，少部分是期间新增
signup_dates = [
    DATE_START - timedelta(days=int(np.random.exponential(scale=30)))
    for _ in range(N_USERS)
]

users = pd.DataFrame({
    'user_id': [f'U{str(i).zfill(4)}' for i in range(N_USERS)],
    'user_type': user_types,
    'signup_date': signup_dates
})

print(f'用户类型分布:\n{users["user_type"].value_counts()}')
print(f'\n前5行:')
users.head()

用户类型分布:
user_type
casual     240
heavy      154
churned    106
Name: count, dtype: int64

前5行:


,user_id,user_type,signup_date
0,U0000,casual,2025-01-25
1,U0001,churned,2025-02-06
2,U0002,casual,2025-02-18
3,U0003,casual,2025-01-10
4,U0004,heavy,2025-01-26


In [3]:
# === 意图场景配置 ===
# 每个场景定义：出现概率、各行为的发生概率

INTENT_CONFIG = {
    'code_generation': {
        'weight': 0.25,  # 占总对话的25%
        'action_probs': {
            'like': 0.15,
            'dislike': 0.15,
            'retry': 0.30,   # 代码场景retry率高
            'followup': 0.25,
            'silent': 0.15
        }
    },
    'knowledge_qa': {
        'weight': 0.25,
        'action_probs': {
            'like': 0.20,
            'dislike': 0.25,  # 知识问答容易出幻觉，点踩多
            'retry': 0.15,
            'followup': 0.20,
            'silent': 0.20
        }
    },
    'creative_writing': {
        'weight': 0.15,
        'action_probs': {
            'like': 0.40,    # 创意写作满意度高
            'dislike': 0.05,
            'retry': 0.20,
            'followup': 0.15,
            'silent': 0.20
        }
    },
    'translation': {
        'weight': 0.15,
        'action_probs': {
            'like': 0.35,
            'dislike': 0.08,
            'retry': 0.12,   # 翻译质量稳定
            'followup': 0.10,
            'silent': 0.35
        }
    },
    'chitchat': {
        'weight': 0.10,
        'action_probs': {
            'like': 0.15,
            'dislike': 0.05,
            'retry': 0.05,
            'followup': 0.10,
            'silent': 0.65   # 闲聊沉默率极高
        }
    },
    'data_analysis': {
        'weight': 0.10,
        'action_probs': {
            'like': 0.18,
            'dislike': 0.18,
            'retry': 0.22,
            'followup': 0.30, # 数据分析追问多
            'silent': 0.12
        }
    }
}

# 校验概率之和
for intent, cfg in INTENT_CONFIG.items():
    total = sum(cfg['action_probs'].values())
    assert abs(total - 1.0) < 1e-6, f'{intent} 行为概率之和 = {total}'

print('意图场景数:', len(INTENT_CONFIG))
for intent, cfg in INTENT_CONFIG.items():
    print(f"  {intent:20s} | 占比 {cfg['weight']:.0%} | 点赞率 {cfg['action_probs']['like']:.0%} | 点踩率 {cfg['action_probs']['dislike']:.0%} | retry率 {cfg['action_probs']['retry']:.0%}")

意图场景数: 6
  code_generation      | 占比 25% | 点赞率 15% | 点踩率 15% | retry率 30%
  knowledge_qa         | 占比 25% | 点赞率 20% | 点踩率 25% | retry率 15%
  creative_writing     | 占比 15% | 点赞率 40% | 点踩率 5% | retry率 20%
  translation          | 占比 15% | 点赞率 35% | 点踩率 8% | retry率 12%
  chitchat             | 占比 10% | 点赞率 15% | 点踩率 5% | retry率 5%
  data_analysis        | 占比 10% | 点赞率 18% | 点踩率 18% | retry率 22%


In [7]:
# === 生成对话记录 ===

# 不同用户类型的活跃度配置
USER_ACTIVITY = {
    'heavy':   {'daily_convs': (3, 8),  'active_days_ratio': (0.7, 0.95)},
    'casual':  {'daily_convs': (1, 3),  'active_days_ratio': (0.15, 0.4)},
    'churned': {'daily_convs': (1, 4),  'active_days_ratio': (0.03, 0.12)}  # 只活跃前几天
}

total_days = (DATE_END - DATE_START).days + 1
intent_names = list(INTENT_CONFIG.keys())
intent_weights = [INTENT_CONFIG[i]['weight'] for i in intent_names]

conversations = []
conv_id = 0
session_id = 0

for _, user in users.iterrows():
    uid = user['user_id']
    utype = user['user_type']
    cfg = USER_ACTIVITY[utype]
    
    # 该用户的活跃天数
    active_ratio = np.random.uniform(*cfg['active_days_ratio'])
    n_active_days = max(1, int(total_days * active_ratio))
    
    # churned用户的活跃日集中在月初
    if utype == 'churned':
        active_dates = sorted(random.sample(
            [DATE_START + timedelta(days=d) for d in range(min(7, total_days))],
            k=min(n_active_days, 7)
        ))
    else:
        active_dates = sorted(random.sample(
            [DATE_START + timedelta(days=d) for d in range(total_days)],
            k=n_active_days
        ))
    
    # 每个活跃日生成若干对话
    for date in active_dates:
        n_convs = np.random.randint(*cfg['daily_convs'])
        session_id += 1
        
        for _ in range(n_convs):
            # 随机时间戳
            hour_probs = [
                0.01, 0.005, 0.005, 0.005, 0.005, 0.005,
                0.02, 0.03, 0.05, 0.07, 0.08, 0.08,
                0.06, 0.06, 0.07, 0.07, 0.06, 0.05,
                0.04, 0.05, 0.06, 0.07, 0.06, 0.035
            ]
            hour_probs = [p / sum(hour_probs) for p in hour_probs]
            hour = np.random.choice(range(24), p=hour_probs)
            minute = np.random.randint(0, 60)
            ts = date.replace(hour=hour, minute=minute)
            
            # 随机意图
            intent = np.random.choice(intent_names, p=intent_weights)
            
            conversations.append({
                'conversation_id': f'C{str(conv_id).zfill(5)}',
                'user_id': uid,
                'session_id': f'S{str(session_id).zfill(5)}',
                'timestamp': ts,
                'intent': intent,
                'query': '',   # 下一步填充
                'response': '' # 下一步填充
            })
            conv_id += 1

conversations = pd.DataFrame(conversations)
print(f'总对话数: {len(conversations)}')
print(f'\n按用户类型统计:')
print(conversations.merge(users[['user_id','user_type']], on='user_id')
      .groupby('user_type')['conversation_id'].count()
      .sort_values(ascending=False))
print(f'\n按意图分布:')
print(conversations['intent'].value_counts(normalize=True).round(3))

总对话数: 22394

按用户类型统计:
user_type
heavy      19162
casual      2835
churned      397
Name: conversation_id, dtype: int64

按意图分布:
intent
code_generation     0.251
knowledge_qa        0.250
translation         0.150
creative_writing    0.148
data_analysis       0.101
chitchat            0.100
Name: proportion, dtype: float64


In [9]:
# === 填充 query 和 response 文本 ===

QUERY_TEMPLATES = {
    'code_generation': [
        '用Python写一个快速排序算法',
        '帮我写一个React登录页面组件',
        'SQL查询：找出每个部门薪资最高的员工',
        '用JavaScript实现一个防抖函数',
        '写一个Python爬虫抓取豆瓣电影Top250',
        '帮我用Go写一个简单的HTTP服务器',
        'pandas怎么对多列进行groupby聚合',
        '用Python实现一个LRU缓存',
        '写一个bash脚本批量重命名文件',
        '帮我写一个Flask RESTful API的CRUD接口',
    ],
    'knowledge_qa': [
        '量子计算和经典计算的本质区别是什么',
        'TCP三次握手的过程详细解释一下',
        '什么是RAG架构，它解决了什么问题',
        'Transformer的注意力机制是怎么工作的',
        '解释一下CAP定理',
        '区块链的共识机制有哪些',
        'HTTP/2相比HTTP/1.1有什么改进',
        'MapReduce的工作原理是什么',
        '什么是零知识证明',
        'BERT和GPT的区别是什么',
    ],
    'creative_writing': [
        '写一首关于秋天的现代诗',
        '帮我写一个科幻短篇故事的开头',
        '用鲁迅的风格写一段对当代社会的评论',
        '写一封辞职信，要委婉但坚定',
        '帮我写一段产品发布会的开场白',
        '写一个悬疑小说的大纲',
        '用幽默的方式写一篇关于加班的吐槽文',
        '帮我写一段婚礼祝词',
    ],
    'translation': [
        '把这段中文翻译成英文：我们的产品已经进入了快速增长期',
        '翻译成日文：明天的会议推迟到下午三点',
        '把这段英文翻译成中文：The quick brown fox jumps over the lazy dog',
        '帮我把这封英文邮件翻译成中文',
        '这句话用更地道的英文怎么说：这个功能还在开发中',
        '翻译成韩文：欢迎来到我们的店铺',
    ],
    'chitchat': [
        '你好呀',
        '今天心情不太好',
        '你觉得人生的意义是什么',
        '推荐一部好看的电影',
        '无聊，跟我聊聊天吧',
        '你有自我意识吗',
        '讲个笑话',
    ],
    'data_analysis': [
        '帮我分析这组销售数据的趋势',
        '怎么用Python做用户留存分析',
        '这个A/B测试的结果显著吗',
        '帮我写一个RFM用户分群的代码',
        '怎么用SQL计算用户的LTV',
        '帮我做一个漏斗分析',
        '这组数据适合用什么统计检验方法',
        '怎么用Python画一个热力图',
    ]
}

# Response模板：大部分是正常回复，埋入一些"问题回复"
RESPONSE_TEMPLATES = {
    'code_generation': {
        'normal': [
            '好的，这是实现代码：\n```python\ndef solution():\n    # 完整实现\n    pass\n```\n代码已经过测试，可以直接使用。',
            '这是完整实现，包含了错误处理和边界条件检查：\n```\n// 代码实现\n```',
            '以下是实现方案，我加了详细注释帮助理解：\n```\n# 步骤1: 初始化\n# 步骤2: 核心逻辑\n# 步骤3: 返回结果\n```',
        ],
        'bad': [
            '抱歉，我无法直接执行代码，但我可以给你一个思路...',
            '这个需求涉及到安全性问题，我建议你参考官方文档来实现。',
            '```python\ndef sort(arr):\n    # 这里用了冒泡排序\n    for i in range(len(arr)):\n        for j in range(i):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n```\n注意这是快速排序的实现。',  # 说是快排实际是冒泡
        ]
    },
    'knowledge_qa': {
        'normal': [
            '这是一个很好的问题。简单来说，核心区别在于以下几点：\n1. 第一点：基本概念\n2. 第二点：技术实现\n3. 第三点：应用场景\n希望这个解释对你有帮助。',
            '让我从底层原理开始解释。首先你需要理解基本概念，然后我们再看具体机制...',
            '这个问题可以从三个维度来理解：理论基础、实际应用和未来发展...',
        ],
        'bad': [
            '根据最新研究，这项技术是由OpenAI在2019年首次提出的，并在2020年获得了图灵奖。',  # 编造事实
            '这个概念最早可以追溯到1985年，由斯坦福大学的Johnson教授在Nature上发表。',  # 虚构人物和论文
            '关于这个问题，业界普遍认为正确答案是A，但实际上根据我的分析应该是B。不过具体细节我不太确定。',  # 含糊其辞
        ]
    },
    'creative_writing': {
        'normal': [
            '这是我为你创作的内容：\n\n[精心构思的创意文本]\n\n如果你希望调整风格或方向，请告诉我。',
            '以下是我的创作，我试图在文字中融入你要求的意境和情感：\n\n[创意内容]\n\n欢迎提出修改意见。',
        ],
        'bad': [
            '这类创作可能涉及版权问题，我建议你自己尝试写作。',
        ]
    },
    'translation': {
        'normal': [
            '翻译如下：\n\n[准确的翻译内容]\n\n这里我选择了更贴合目标语言习惯的表达方式。',
            '这是翻译结果：\n[翻译内容]\n\n注意：其中"XX"一词我采用了意译而非直译，这样更自然。',
        ],
        'bad': [
            '翻译结果：[包含语法错误的翻译]\n\n（注：部分专业术语可能需要人工校对）',
        ]
    },
    'chitchat': {
        'normal': [
            '你好！今天有什么我可以帮你的吗？',
            '我理解你的感受。虽然我是AI，但我很愿意听你说说。',
            '这是个很有深度的问题，让我认真想想...',
        ],
        'bad': [
            '作为一个AI语言模型，我没有情感和个人观点。我只能基于训练数据来回答问题。',
        ]
    },
    'data_analysis': {
        'normal': [
            '好的，让我帮你分析一下。首先看数据的基本分布：\n[分析过程]\n结论是...',
            '这是完整的分析代码和结果解读：\n```python\n# 数据分析代码\n```\n从结果来看...',
        ],
        'bad': [
            '你需要先提供具体的数据集，我才能帮你分析。请把数据发给我。',
            '这个分析比较复杂，建议你使用Excel的数据透视表功能来完成。',
        ]
    }
}

# 填充：bad response的概率跟该场景的dislike+retry率正相关
def fill_text(row):
    intent = row['intent']
    query = random.choice(QUERY_TEMPLATES[intent])
    
    bad_prob = (INTENT_CONFIG[intent]['action_probs']['dislike'] + 
                INTENT_CONFIG[intent]['action_probs']['retry']) * 0.8
    
    if np.random.random() < bad_prob:
        response = random.choice(RESPONSE_TEMPLATES[intent]['bad'])
    else:
        response = random.choice(RESPONSE_TEMPLATES[intent]['normal'])
    
    return pd.Series({'query': query, 'response': response})

conversations[['query', 'response']] = conversations.apply(fill_text, axis=1)

print('文本填充完成')
print(f'\n随机看3条:')
for _, row in conversations.sample(3, random_state=SEED).iterrows():
    print(f"\n[{row['intent']}] Q: {row['query'][:40]}...")
    print(f"  A: {row['response'][:60]}...")

文本填充完成

随机看3条:

[code_generation] Q: SQL查询：找出每个部门薪资最高的员工...
  A: 以下是实现方案，我加了详细注释帮助理解：
```
# 步骤1: 初始化
# 步骤2: 核心逻辑
# 步骤3: 返回结果
...

[knowledge_qa] Q: Transformer的注意力机制是怎么工作的...
  A: 根据最新研究，这项技术是由OpenAI在2019年首次提出的，并在2020年获得了图灵奖。...

[data_analysis] Q: 怎么用Python做用户留存分析...
  A: 这是完整的分析代码和结果解读：
```python
# 数据分析代码
```
从结果来看......


In [10]:
# === 生成事件流 ===

# 追问文本模板
FOLLOWUP_TEMPLATES = {
    'exploratory': [  # 探索式追问——用户满意，想深入
        '能再详细说说吗',
        '这个方案的优缺点是什么',
        '还有其他方案吗',
        '如果数据量很大的话怎么优化',
        '能举个具体的例子吗',
        '这个和XX有什么区别',
        '实际项目中一般怎么选择',
        '有没有推荐的学习资料',
        '性能方面怎么样',
        '能再展开讲讲第二点吗',
    ],
    'corrective': [  # 纠错式追问——用户不满意，在纠正
        '不是，我问的是具体实现，不是概念',
        '你理解错了，我要的是Python不是Java',
        '这个答案有问题吧，我查到的不是这样',
        '能不能直接给代码，不要光说思路',
        '你没有回答我的问题',
        '这不对，正确的应该是...',
        '我说的不是这个意思，我的需求是...',
        '太笼统了，能不能具体一点',
        '你前面说的和现在说的矛盾了',
        '这个结果明显不对啊',
    ]
}

events = []
event_id = 0

for _, conv in conversations.iterrows():
    cid = conv['conversation_id']
    intent = conv['intent']
    ts = conv['timestamp']
    probs = INTENT_CONFIG[intent]['action_probs']
    
    # 决定第一个行为
    first_action = np.random.choice(
        list(probs.keys()), p=list(probs.values())
    )
    
    if first_action == 'silent':
        # 沉默用户不产生任何事件
        continue
    
    # 生成第一个事件
    event_ts = ts + timedelta(seconds=np.random.randint(5, 120))
    followup_text = ''
    
    if first_action == 'followup':
        # 纠错式追问的比例跟该场景的dislike率正相关
        corrective_prob = min(0.7, probs['dislike'] * 2.5)
        if np.random.random() < corrective_prob:
            followup_text = random.choice(FOLLOWUP_TEMPLATES['corrective'])
        else:
            followup_text = random.choice(FOLLOWUP_TEMPLATES['exploratory'])
    
    events.append({
        'event_id': f'E{str(event_id).zfill(6)}',
        'conversation_id': cid,
        'event_type': first_action,
        'event_timestamp': event_ts,
        'followup_text': followup_text
    })
    event_id += 1
    
    # 部分对话会有第二个行为（比如retry之后又点赞/点踩）
    if first_action == 'retry' and np.random.random() < 0.4:
        second_action = np.random.choice(['like', 'dislike'], p=[0.6, 0.4])
        event_ts2 = event_ts + timedelta(seconds=np.random.randint(10, 180))
        events.append({
            'event_id': f'E{str(event_id).zfill(6)}',
            'conversation_id': cid,
            'event_type': second_action,
            'event_timestamp': event_ts2,
            'followup_text': ''
        })
        event_id += 1
    
    # 追问之后也可能点赞或点踩
    if first_action == 'followup' and np.random.random() < 0.3:
        second_action = 'like' if 'exploratory' in followup_text or np.random.random() < 0.5 else 'dislike'
        event_ts2 = event_ts + timedelta(seconds=np.random.randint(30, 300))
        events.append({
            'event_id': f'E{str(event_id).zfill(6)}',
            'conversation_id': cid,
            'event_type': second_action,
            'event_timestamp': event_ts2,
            'followup_text': ''
        })
        event_id += 1

events = pd.DataFrame(events)

print(f'总事件数: {len(events)}')
print(f'\n事件类型分布:')
print(events['event_type'].value_counts())
print(f'\n有追问文本的事件数: {(events["followup_text"] != "").sum()}')
print(f'\n多事件对话示例:')
multi = events.groupby('conversation_id').filter(lambda x: len(x) > 1)
sample_cid = multi['conversation_id'].iloc[0]
print(events[events['conversation_id'] == sample_cid][['event_type', 'event_timestamp', 'followup_text']])

总事件数: 19952

事件类型分布:
event_type
like        6948
dislike     4471
followup    4335
retry       4198
Name: count, dtype: int64

有追问文本的事件数: 4335

多事件对话示例:
  event_type     event_timestamp followup_text
1      retry 2025-03-06 14:15:17              
2    dislike 2025-03-06 14:16:39              


In [11]:
# === 保存数据 ===

os.makedirs(OUTPUT_DIR, exist_ok=True)

users.to_csv(f'{OUTPUT_DIR}/users.csv', index=False)
conversations.to_csv(f'{OUTPUT_DIR}/conversations.csv', index=False)
events.to_csv(f'{OUTPUT_DIR}/events.csv', index=False)

print('文件已保存:')
for f in ['users.csv', 'conversations.csv', 'events.csv']:
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'  {f}: {size/1024:.1f} KB')

print(f'\n数据概览:')
print(f'  用户数: {len(users)}')
print(f'  对话数: {len(conversations)}')
print(f'  事件数: {len(events)}')

文件已保存:
  users.csv: 12.2 KB
  conversations.csv: 4569.4 KB
  events.csv: 991.6 KB

数据概览:
  用户数: 500
  对话数: 22394
  事件数: 19952
